# Infer — AITeamVN/Vi-Qwen2-3B-RAG + adapter LoRA, test 1.000 câu

`prompts.jsonl` gắn sẵn (lượt 3 BM25+rerank cho adapter phiên 3; hybrid của B cho adapter phiên 5):
prompt test phải cùng kiểu retrieval với dữ liệu train adapter. Chia shard `TU/DEN`, merge ở máy dev.


In [ ]:
# ORCHESTRA-PARAMS
TU, DEN = 0, 500


In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
os.environ["PYTHONPATH"] = "src"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())
# Dataset tạo version bất đồng bộ: bản cũ sinh lại cả 1.000 câu mỗi shard và OOM math-kernel.
_s = (REPO / "scripts/infer.py").read_text()
assert "use_gqa_in_sdpa" in _s and "ids: set[str]" in _s, "codebase cũ — chờ legalqa-v10-src tạo xong version"

!pip -q install transformers==5.14.1 peft nltk huggingface_hub
!pip -q uninstall -y torchao


In [ ]:
CAU_HOI = next(Path("/kaggle/input").glob("**/public-official.json"), None)
PROMPTS = next(Path("/kaggle/input").glob("**/prompts.jsonl"), None)
assert CAU_HOI and PROMPTS, "gắn dataset legalqa-test-prompts-phien3"
assert len(json.loads(CAU_HOI.read_text(encoding="utf-8"))) == 1000
_p = [json.loads(l) for l in open(PROMPTS, encoding="utf-8")]
assert len({x["id"] for x in _p}) == 1000, "prompts.jsonl phải đủ 1.000 câu"

RA = Path("/kaggle/working/out"); RA.mkdir(parents=True, exist_ok=True)
shutil.copy(PROMPTS, RA / "prompts.jsonl")

# /tmp chứ không /kaggle/working: 6 GB trọng số không được lọt vào output kernel.
from huggingface_hub import snapshot_download
BASE = snapshot_download("AITeamVN/Vi-Qwen2-3B-RAG", revision="eaf427c24d86066a2b35828c499b7db3af321227",
                         local_dir="/tmp/base", allow_patterns=["*.json", "*.safetensors", "*.txt"])
# Mỗi kernel chỉ gắn MỘT adapter — hai tot_nhat/ là không biết đang infer bản nào.
_ad = [q.parent for q in Path("/kaggle/input").glob("**/tot_nhat/adapter_config.json")]
assert len(_ad) == 1, f"cần đúng 1 adapter tot_nhat/, thấy {_ad}"
ADAPTER = _ad[0]
_ac = json.loads((ADAPTER / "adapter_config.json").read_text())
assert "Vi-Qwen2" in str(_ac.get("base_model_name_or_path")) or "/tmp/base" in str(_ac.get("base_model_name_or_path")), \
    f"adapter train trên model khác: {_ac.get('base_model_name_or_path')}"
print("shard", TU, DEN, "| base", BASE, "| adapter", ADAPTER)


In [ ]:
# --gio-pha2 10.5: 500 câu, tốc độ 3B chưa đo (2B + adapter ~30 s/câu). Hết giờ thì
# du_doan.jsonl vẫn giữ phần đã sinh, không ghi mốc; chạy lại shard là nối tiếp.
subprocess.run([sys.executable, "scripts/infer.py",
    "--cau-hoi", str(CAU_HOI), "--model", BASE, "--adapter", str(ADAPTER),
    "--ra", str(RA), "--bo-qua-pha1", "--tu", str(TU), "--den", str(DEN),
    "--max-new", "1024", "--gen-bs", "2", "--gio-pha2", "10.5",
], check=True)
n = sum(1 for _ in open(RA / "du_doan.jsonl", encoding="utf-8"))
print(f"du_doan: {n}/{DEN-TU}")
